# Задача 11. Возвраты, отмены и обращения в поддержку

**Постановка задачи:** нужно оценить проблемные категории товаров. Для каждой категории посчитать:

- количество заказов с этой категорией;
- количество возвратов;
- количество отмен;
- количество обращений в поддержку;
- среднюю оценку удовлетворённости по тикетам.

Результат отсортировать по доле возвратов.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH order_categories AS (
    SELECT DISTINCT
        o.order_id,
        o.status,
        p.category
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
), category_tickets AS (
    SELECT
        oc.category,
        COUNT(st.ticket_id) AS tickets,
        AVG(st.satisfaction_score) AS avg_satisfaction_score
    FROM order_categories oc
    LEFT JOIN support_tickets st ON oc.order_id = st.order_id
    GROUP BY oc.category
)
SELECT
    oc.category,
    COUNT(oc.order_id) AS category_orders,
    SUM(CASE WHEN oc.status = 'refunded' THEN 1 ELSE 0 END) AS refunded_orders,
    SUM(CASE WHEN oc.status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
    ROUND(100.0 * SUM(CASE WHEN oc.status = 'refunded' THEN 1 ELSE 0 END) / COUNT(oc.order_id), 2) AS refund_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN oc.status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(oc.order_id), 2) AS cancel_rate_pct,
    ct.tickets,
    ROUND(ct.avg_satisfaction_score, 2) AS avg_satisfaction_score
FROM order_categories oc
JOIN category_tickets ct ON oc.category = ct.category
GROUP BY oc.category, ct.tickets, ct.avg_satisfaction_score
ORDER BY refund_rate_pct DESC, category_orders DESC;
"""

df = pd.read_sql_query(query, conn)
df

**Комментарий стажёра:** одна покупка может содержать товары из нескольких категорий, поэтому сначала сделал `DISTINCT order_id + category`, чтобы не раздуть число заказов товарными строками.